# AutoMixAI – Training v2: GTZAN + Hainsworth + Richer Features

**Improvements over v1:**
- ✅ **GTZAN** dataset: 1,000 tracks × 30s, 10 genres — good genre diversity
- ✅ **Hainsworth** dataset: 222 tracks with **real human-annotated beat timestamps** (much better than pseudo-labels)
- ✅ **Richer features** (50 dims): 13 MFCCs + 13 delta-MFCCs + 12 chroma + spectral centroid + rolloff + ZCR + onset + flux + energy
- ✅ **Deeper model**: Dense(256) → Dense(128) → Dense(64) → Dense(1)
- ✅ **Class-weight balancing**: compensates for beat sparsity (~5% positive)
- ✅ **Early stopping** + model checkpoint

**Output:** `beat_detector_v2.h5` → rename to `beat_detector.h5` and place in `backend/app/storage/models/`

---

### Kaggle Dataset Setup

Add these as Kaggle Datasets before running:

| Dataset | Kaggle slug | Notes |
|---|---|---|
| GTZAN | `andradaolteanu/gtzan-dataset-music-genre-classification` | Public, 1.2 GB |
| Hainsworth | Upload from `download_hainsworth.py` output | ~1.8 GB |
| Ballroom (optional) | Your existing Kaggle dataset | Adds more WAV data |

## 1. Install Dependencies

In [ ]:
!pip install librosa soundfile tensorflow numpy scipy mirdata -q
import warnings; warnings.filterwarnings('ignore')
print('✅ Dependencies installed')

## 2. Configuration

In [ ]:
from pathlib import Path

# ── Dataset paths (adjust to your Kaggle dataset names) ──────────────
GTZAN_DIR       = Path('/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original')
HAINSWORTH_DIR  = Path('/kaggle/input/hainsworth-dataset')   # your uploaded Hainsworth folder
BALLROOM_DIR    = Path('/kaggle/input/ballroom-dataset/BallroomData')  # optional

# ── Audio parameters ─────────────────────────────────────────────────
SAMPLE_RATE  = 22050
HOP_LENGTH   = 512       # training hop (fine-grained labels)
N_FFT        = 2048
N_MFCC       = 13

# ── Training parameters ──────────────────────────────────────────────
EPOCHS       = 80
BATCH_SIZE   = 512       # larger batch = faster epochs on Kaggle GPU
PATIENCE     = 10        # early stopping patience

# ── File limits (set None for full dataset) ──────────────────────────
MAX_GTZAN       = None   # 1000 tracks x 30s each
MAX_HAINSWORTH  = None   # 222 annotated tracks
MAX_BALLROOM    = None   # 698 WAV files

print('Configuration loaded.')
print(f'  Feature dimensions: {N_MFCC} MFCCs + {N_MFCC} delta-MFCCs + 12 chroma + 5 spectral = {N_MFCC*2 + 12 + 5} total')

## 3. Richer Feature Extraction (50-dim)

In [ ]:
import numpy as np
import librosa

FEATURE_DIM = N_MFCC * 2 + 12 + 5   # 13 MFCC + 13 delta + 12 chroma + 5

def load_audio(path, sr=SAMPLE_RATE):
    """Load audio, convert to mono, normalize to [-1, 1]."""
    y, _ = librosa.load(str(path), sr=sr, mono=True)
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    return y, sr


def extract_features(y, sr):
    """
    Extract 50-dimensional feature vector per frame.

    Features:
      [0:13]  MFCCs
      [13:26] Delta-MFCCs   (velocity of MFCCs — captures beat impulse)
      [26:38] Chroma STFT   (harmonic/key content)
      [38]    Onset strength
      [39]    Spectral flux
      [40]    RMS energy
      [41]    Spectral centroid (brightness)
      [42]    ZCR             (noisiness)
    """
    hop = HOP_LENGTH
    n_fft = N_FFT

    # Single STFT pass — reused across all features
    S_complex = librosa.stft(y, n_fft=n_fft, hop_length=hop)
    S_mag = np.abs(S_complex)
    n_frames = S_mag.shape[1]

    # MFCCs via mel-spectrogram (reuses S_mag)
    mel = librosa.feature.melspectrogram(S=S_mag**2, sr=sr, n_fft=n_fft)
    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel), n_mfcc=N_MFCC)   # (13, T)
    delta_mfcc = librosa.feature.delta(mfcc)                                   # (13, T)

    # Chroma (reuses S_mag)
    chroma = librosa.feature.chroma_stft(S=S_mag**2, sr=sr, hop_length=hop, n_fft=n_fft)  # (12, T)

    # Onset strength (from db-scaled S_mag)
    onset = librosa.onset.onset_strength(
        S=librosa.amplitude_to_db(S_mag, ref=np.max), sr=sr, hop_length=hop
    )  # (T,)

    # Spectral flux
    flux = np.sqrt(np.sum(np.diff(S_mag, axis=1) ** 2, axis=0))
    flux = np.concatenate([[0.0], flux])  # (T,)

    # RMS energy
    rms = librosa.feature.rms(S=S_mag, frame_length=n_fft, hop_length=hop)[0]  # (T,)

    # Spectral centroid (brightness proxy)
    centroid = librosa.feature.spectral_centroid(S=S_mag, sr=sr, hop_length=hop, n_fft=n_fft)[0]  # (T,)

    # Zero-crossing rate
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=n_fft, hop_length=hop)[0]  # (T,)

    # Align all to common frame count
    T = min(n_frames, mfcc.shape[1], chroma.shape[1], len(onset), len(flux), len(rms), len(centroid), len(zcr))

    features = np.column_stack([
        mfcc.T[:T],           # 13
        delta_mfcc.T[:T],     # 13
        chroma.T[:T],         # 12
        onset[:T],            # 1
        flux[:T],             # 1
        rms[:T],              # 1
        centroid[:T],         # 1
        zcr[:T],              # 1
    ])  # (T, 43)
    return features


def pseudo_beat_labels(y, sr, n_frames):
    """Generate beat labels using librosa (used for GTZAN — no annotations)."""
    _, beat_frames = librosa.beat.beat_track(y=y, sr=sr, hop_length=HOP_LENGTH)
    labels = np.zeros(n_frames, dtype=np.float32)
    for bf in beat_frames:
        if 0 <= bf < n_frames:
            labels[bf] = 1.0
    return labels


def annotation_beat_labels(beat_times, sr, n_frames):
    """Convert real beat timestamps → frame labels (used for Hainsworth)."""
    labels = np.zeros(n_frames, dtype=np.float32)
    beat_frames = librosa.time_to_frames(beat_times, sr=sr, hop_length=HOP_LENGTH)
    for bf in beat_frames:
        if 0 <= bf < n_frames:
            labels[bf] = 1.0
    return labels


# Quick sanity-check
y_test = np.random.randn(sr := SAMPLE_RATE).astype(np.float32)
f_test = extract_features(y_test, sr)
print(f'Feature shape for 1s of audio: {f_test.shape}  (frames × features)')
print(f'✅ Feature extractor OK — {f_test.shape[1]}-dimensional')

## 4. Process GTZAN Dataset (pseudo-labels)

In [ ]:
all_X, all_y = [], []

if GTZAN_DIR.exists():
    audio_files = sorted(GTZAN_DIR.rglob('*.wav'))
    if MAX_GTZAN:
        audio_files = audio_files[:MAX_GTZAN]
    print(f'GTZAN: processing {len(audio_files)} WAV files across {len(list(GTZAN_DIR.iterdir()))} genres...')

    ok = skip = 0
    for i, path in enumerate(audio_files, 1):
        try:
            y_audio, sr = load_audio(path)
            feats = extract_features(y_audio, sr)
            labels = pseudo_beat_labels(y_audio, sr, feats.shape[0])
            all_X.append(feats)
            all_y.append(labels)
            ok += 1
        except Exception as e:
            skip += 1
            if skip <= 5:
                print(f'  SKIP {path.name}: {e}')
        if i % 100 == 0:
            print(f'  [{i}/{len(audio_files)}] ok={ok} skip={skip}')

    print(f'GTZAN done ✅  ok={ok}  skipped={skip}  cumulative tracks={len(all_X)}')
else:
    print(f'⚠️  GTZAN not found at {GTZAN_DIR} — skipping')

## 5. Process Hainsworth Dataset (REAL beat annotations)

In [ ]:
if HAINSWORTH_DIR.exists():
    import mirdata
    print('Loading Hainsworth via mirdata...')
    hainsworth = mirdata.initialize('hainsworth', data_home=str(HAINSWORTH_DIR))
    track_ids = hainsworth.track_ids
    if MAX_HAINSWORTH:
        track_ids = track_ids[:MAX_HAINSWORTH]

    print(f'Hainsworth: processing {len(track_ids)} annotated tracks...')
    ok = skip = 0

    for i, tid in enumerate(track_ids, 1):
        try:
            track = hainsworth.track(tid)
            audio_path = track.audio_path
            beats_obj = track.beats

            if beats_obj is None or beats_obj.times is None or len(beats_obj.times) == 0:
                skip += 1
                continue

            y_audio, sr = load_audio(audio_path)
            feats = extract_features(y_audio, sr)

            # Use REAL annotation timestamps — much better than pseudo-labels!
            labels = annotation_beat_labels(beats_obj.times, sr, feats.shape[0])
            all_X.append(feats)
            all_y.append(labels)
            ok += 1

        except Exception as e:
            skip += 1
            if skip <= 5:
                print(f'  SKIP {tid}: {e}')

        if i % 50 == 0:
            print(f'  [{i}/{len(track_ids)}] ok={ok} skip={skip}')

    print(f'Hainsworth done ✅  ok={ok}  skipped={skip}  cumulative tracks={len(all_X)}')
else:
    print(f'⚠️  Hainsworth not found at {HAINSWORTH_DIR} — skipping')

## 6. Process Ballroom Dataset (optional, pseudo-labels)

In [ ]:
if BALLROOM_DIR.exists():
    wav_files = sorted(BALLROOM_DIR.rglob('*.wav'))
    if MAX_BALLROOM:
        wav_files = wav_files[:MAX_BALLROOM]
    print(f'Ballroom: processing {len(wav_files)} WAV files...')

    ok = skip = 0
    for i, path in enumerate(wav_files, 1):
        try:
            y_audio, sr = load_audio(path)
            feats = extract_features(y_audio, sr)
            labels = pseudo_beat_labels(y_audio, sr, feats.shape[0])
            all_X.append(feats)
            all_y.append(labels)
            ok += 1
        except Exception as e:
            skip += 1
        if i % 100 == 0:
            print(f'  [{i}/{len(wav_files)}] ok={ok} skip={skip}')

    print(f'Ballroom done ✅  ok={ok}  skipped={skip}  cumulative={len(all_X)}')
else:
    print('Ballroom not found — skipping (optional)')

## 7. Combine & Prepare

In [ ]:
assert len(all_X) > 0, '❌ No data loaded! Check your dataset paths above.'

X = np.vstack(all_X).astype(np.float32)
y = np.concatenate(all_y).astype(np.float32)

print(f'Combined dataset:')
print(f'  Features (X) : {X.shape}')
print(f'  Labels   (y) : {y.shape}')
print(f'  Beat ratio   : {y.mean():.4f}  ({int(y.sum())} beats / {len(y)} frames)')

# ── Feature normalisation (zero-mean, unit-variance) ──────────────────
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)
print(f'\nFeatures normalised (StandardScaler).')

# Save scaler params for inference
import joblib
joblib.dump(scaler, '/kaggle/working/feature_scaler.pkl')
print('Scaler saved → /kaggle/working/feature_scaler.pkl')

# Save processed arrays
np.save('/kaggle/working/X.npy', X)
np.save('/kaggle/working/y.npy', y)
print('Saved X.npy, y.npy')

# ── Class weights to compensate for beat sparsity ─────────────────────
beat_ratio = y.mean()
class_weight = {0: 1.0, 1: (1 - beat_ratio) / beat_ratio}
print(f'\nClass weights: non-beat=1.0  beat={class_weight[1]:.1f}')

## 8. Build Deeper ANN Model

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

input_dim = X.shape[1]

model = keras.Sequential([
    layers.Input(shape=(input_dim,)),

    # Block 1
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    # Block 2
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    # Block 3
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),

    # Output
    layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')],
)

model.summary()
print(f'\nInput dim : {input_dim}')
print(f'Parameters: {model.count_params():,}')

## 9. Train with Early Stopping & LR Scheduling

In [ ]:
callbacks = [
    # Stop early if val_loss doesn't improve
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE,
        restore_best_weights=True, verbose=1,
    ),
    # Reduce LR on plateau
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4,
        min_lr=1e-6, verbose=1,
    ),
    # Save best checkpoint
    keras.callbacks.ModelCheckpoint(
        '/kaggle/working/beat_detector_v2_best.h5',
        monitor='val_auc', mode='max',
        save_best_only=True, verbose=0,
    ),
]

history = model.fit(
    X, y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.15,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

print('\n── Final metrics ──────────────────────────────────')
for metric in ['accuracy', 'val_accuracy', 'auc', 'val_auc', 'precision', 'recall']:
    if metric in history.history:
        print(f'  {metric:20s}: {history.history[metric][-1]:.4f}')

## 10. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

pairs = [
    ('accuracy', 'val_accuracy', 'Accuracy'),
    ('loss',     'val_loss',     'Loss'),
    ('auc',      'val_auc',      'AUC'),
    ('precision','recall',       'Precision vs Recall (train)'),
]

for ax, (m1, m2, title) in zip(axes.flat, pairs):
    if m1 in history.history:
        ax.plot(history.history[m1], label=m1.split('_')[-1].capitalize())
    if m2 in history.history:
        ax.plot(history.history[m2], label=m2.split('_')[-1].capitalize(), linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves_v2.png', dpi=150)
plt.show()
print('Saved training_curves_v2.png')

## 11. Save Final Model & Files

In [ ]:
model.save('/kaggle/working/beat_detector_v2.h5')
print('✅ Model saved → /kaggle/working/beat_detector_v2.h5')
print('✅ Scaler saved → /kaggle/working/feature_scaler.pkl')

# Feature dim — needed to update backend config
print(f'\nFeature dimension: {X.shape[1]}')
print(f'(Update N_DIMS in backend/app/utils/config.py if different from 16)')

print('\n=== DOWNLOAD THESE FILES FROM OUTPUT TAB ===')
print('  1. beat_detector_v2.h5       → rename to beat_detector.h5')
print('     place in: backend/app/storage/models/')
print('  2. feature_scaler.pkl        → place in: backend/app/storage/models/')
print('     (update inference.py to apply scaler before prediction)')
print('  3. training_curves_v2.png    → review training health')